In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display, Image, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "analysis/reporting/make_report.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open blv-project-team as your VS Code folder, then reopen this notebook.")
REPORT_DIR = PROJECT_ROOT / "analysis/reporting"
OUTPUT_DIR = REPORT_DIR / "report_outputs"
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
print("Report output:", OUTPUT_DIR)


In [ ]:
result = subprocess.run(
    [sys.executable, str(REPORT_DIR / "make_report.py"), "--project-root", str(PROJECT_ROOT)],
    text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


In [ ]:
main_tables = {
    "Dataset composition": "01_dataset",
    "Blur heuristic performance": "03_blur_performance",
    "Analysis coverage": "05_coverage",
    "Human scores by model and image condition": "07_human_model_scores",
    "Mean of six reviewer-pair agreement values": "10_team_agreement",
    "Spearman and Kendall correlations": "11_correlations",
}
for title, filename in main_tables.items():
    display(Markdown("### " + title))
    table = pd.read_csv(OUTPUT_DIR / "tables" / (filename + ".csv"))
    display(table.style.format(precision=4, na_rep="N/A"))


In [ ]:
import json
manifest = json.loads((OUTPUT_DIR / "report_manifest.json").read_text())
for figure in manifest["figures"]:
    if figure["section"] == "Main":
        display(Markdown("### " + figure["title"]))
        display(Image(filename=str(OUTPUT_DIR / "figures" / (figure["name"] + ".png")), width=1000))
        display(Markdown(figure["note"]))


In [ ]:
print("Gallery:", OUTPUT_DIR / "index.html")
print("Tables:", OUTPUT_DIR / "tables")
print("Figures:", OUTPUT_DIR / "figures")
print("Captions:", OUTPUT_DIR / "CAPTIONS.md")
